# Clinical Note Triage — starter notebook

Read a free-text **medical transcription** and predict which **clinical specialty**
it belongs to — automatic triage of clinical notes, across **12 specialties**
(`spec_01` … `spec_12`).

This notebook runs end-to-end: **download the data → train a simple model → submit**.
Metric: **F1-macro**.

> Open in Colab, then run the cells top to bottom. The only thing you must edit is
> your API token in the next cell (find it on your **Profile** page on ml-arena.com).

## 0. Setup

In [ ]:
!pip install -q mlarena-sdk scikit-learn pandas  # installs the `mlarena` package

import mlarena
import pandas as pd

API_TOKEN = "mlk_user_REPLACE_ME"   # <-- paste your token (Profile page)
COMPETITION_ID = 174

client = mlarena.connect(api_key=API_TOKEN, base_url="https://ml-arena.com")

## 1. Get the data

`download_dataset` pulls the competition files into `data/`. Each row is one free-text
clinical transcription; the training labels are in `train.csv` (`id,transcription,label`).

In [ ]:
client.download_dataset(COMPETITION_ID, "data/")

train = pd.read_csv("data/train.csv")   # id, transcription, label
test  = pd.read_csv("data/test.csv")    # id, transcription

print("train:", train.shape, "  test:", test.shape)
print(train["label"].value_counts())
train.head(2)

## 2. A baseline model

Turn each transcription into **TF-IDF** features (word + bigram counts, down-weighted by
how common they are) and fit a logistic regression. This is just a starting point — word
embeddings or a fine-tuned transformer will score much higher. We use 3-fold
cross-validation to estimate the F1-macro before submitting.

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.model_selection import cross_val_score

model = Pipeline([
    ("tfidf", TfidfVectorizer(sublinear_tf=True, ngram_range=(1, 2),
                              min_df=3, max_features=40000, stop_words="english")),
    ("clf", LogisticRegression(max_iter=2000, C=5.0, class_weight="balanced")),
])

cv = cross_val_score(model, train["transcription"], train["label"], cv=3, scoring="f1_macro")
print("CV F1-macro: %.4f +/- %.4f" % (cv.mean(), cv.std()))

model.fit(train["transcription"], train["label"])   # refit on all the training data

## 3. Predict and write `submission.csv`

In [ ]:
pred = model.predict(test["transcription"])
submission = pd.DataFrame({"id": test["id"], "label": pred})
submission.to_csv("submission.csv", index=False)
submission.head()

## 4. Submit

Run the cell below to submit through the SDK, or upload `submission.csv` on the
competition page.

In [ ]:
result = client.submit(competition_id=COMPETITION_ID, files=["submission.csv"])
print(result)